# 04 — Statistical & Temporal Feature Extraction

Complements notebook 03's frequency-domain features with classical time-domain descriptors, computed on the same 60s/30s windows.

## Design decisions

| Decision | Choice | Why |
|---|---|---|
| Windowing | Same 60s windows, 30s step as notebook 03 (`get_phase_windows`, reused via `assemble_subject_rows`) | Rows must line up with `frequency_features.csv` for a later merge -- same window/step, same phase-boundary rules, same mini-EmotionalStress auto-drop. |
| Features per signal | `mean`, `std`, `skewness`, `slope` | Mean/std capture overall level and spread; skewness captures whether a burst rides on a flat baseline vs. a symmetric spread; slope (least-squares fit against time) captures whether the signal is rising or falling right now -- a direct stress marker (e.g. rising EDA) that a spectrum doesn't state as plainly. |
| Applied to | All five signals (EDA, Acc magnitude, Temp, HR, SpO2), including HR/SpO2 at 60 raw samples/window | These are plain statistics, not spectral estimates -- no frequency-resolution-vs-sample-count tradeoff like notebook 03 had, so there's no reason to treat any signal differently. |
| HR gets six extra: `rms`, `std`, `min`, `max`, `median`, `slope` of `diff(hr)` | Successive-difference statistics on the second-by-second HR series | Closest available proxy for heart-rate variability. Canonical HRV (RMSSD, SDNN, ...) needs beat-to-beat RR intervals from a raw PPG/ECG waveform, and this dataset only has HR pre-averaged to 1 Hz bpm with no raw PPG channel to detect systolic peaks from (same limitation already noted for `hr_lf_power`/`hr_hf_power` in notebook 03) -- these six statistics on the first-difference series are the nearest reachable substitute. |

Output: `outputs/processed/statistical_features.csv`, then merged with notebook 03's `frequency_features.csv` into `outputs/processed/combined_features.csv`.

In [1]:
import sys
sys.path.insert(0, '..')

from pathlib import Path

import pandas as pd

from src.preprocessing import assemble_subject_rows, load_and_preprocess_subject
from src.statistical_features import extract_window_statistical_features

In [2]:
WINDOW_SECONDS = 60
STEP_SECONDS = 30

## Single-subject demo -- Subject 1

In [3]:
record_acc_temp_eda, record_spo2_hr = load_and_preprocess_subject(
    1, accel_cutoff=None, eda_temp_cutoff=1.0, spo2_hr_cutoff=None
)

demo_rows = assemble_subject_rows(
    record_acc_temp_eda, record_spo2_hr, WINDOW_SECONDS, STEP_SECONDS, extract_window_statistical_features
)
demo_features_df = pd.DataFrame(demo_rows)
demo_features_df.head()

,window_index,label,start_time,acc_mean,acc_std,acc_skewness,acc_slope,eda_mean,eda_std,eda_skewness,...,hr_successive_diff_rms,hr_successive_diff_std,hr_successive_diff_min,hr_successive_diff_max,hr_successive_diff_median,hr_successive_diff_slope,spo2_mean,spo2_std,spo2_skewness,spo2_slope
0,0,Relax,0.0,0.048230,0.042612,1.861465,-0.001234,-0.912600,0.001168,-0.054061,...,0.082808,0.077708,-0.281335,0.281335,0.0,0.000071,0.758037,0.641601,-0.561173,-0.002915
1,1,Relax,30.0,0.946558,0.179161,0.236903,0.007760,-0.909901,0.002445,0.385739,...,0.086334,0.082503,-0.187586,0.187586,0.0,0.001003,0.777040,0.735721,0.124094,-0.004087
2,2,Relax,60.0,0.309399,0.438908,2.863612,-0.014535,-0.907164,0.001510,-0.803237,...,0.087194,0.086831,-0.187586,0.187586,0.0,0.000970,0.282973,0.890287,0.483773,-0.030824
3,3,Relax,90.0,0.028892,0.009842,1.268396,-0.000117,-0.905964,0.000822,-1.024976,...,0.067982,0.067963,-0.187586,0.187586,0.0,-0.000581,-0.116082,0.504195,-1.055290,0.013939
4,4,Relax,120.0,0.026720,0.007728,1.490781,-0.000026,-0.905401,0.000304,-3.205471,...,0.054601,0.053761,-0.187586,0.093837,0.0,0.000855,-0.078076,0.482231,-1.260977,-0.013179


### Sanity check -- do features separate Relax from stress phases?

Mean of a few key features per phase. Expect `eda_slope` to read near flat/negative in Relax (decaying arousal) and clearly positive during stress onset; `acc_std` to be much higher in PhysicalStress than Relax.

In [4]:
demo_features_df.groupby('label')[['eda_mean', 'eda_std', 'eda_slope', 'acc_std', 'hr_mean']].mean()

,eda_mean,eda_std,eda_slope,acc_std,hr_mean
label,,,,,
CognitiveStress,1.195795,0.222023,-0.004859,0.221546,0.123129
EmotionalStress,-0.450626,0.035704,-0.000567,0.202611,-0.283957
PhysicalStress,-0.864859,0.007394,0.000380,0.457631,1.137083
Relax,-0.050411,0.116606,-0.003886,0.135010,-0.308283


## Full dataset -- extract features for all 20 subjects

In [5]:
all_rows = []
for subject_id in range(1, 21):
    try:
        record_acc, record_hr = load_and_preprocess_subject(
            subject_id, accel_cutoff=None, eda_temp_cutoff=1.0, spo2_hr_cutoff=None
        )
        subject_rows = assemble_subject_rows(
            record_acc, record_hr, WINDOW_SECONDS, STEP_SECONDS, extract_window_statistical_features
        )
        for row in subject_rows:
            row['subject_id'] = subject_id
        all_rows.extend(subject_rows)
    except Exception as error:
        print(f"Subject {subject_id}: skipped due to error: {error}")

statistical_features_df = pd.DataFrame(all_rows)
print(f"Full dataset: {statistical_features_df.shape[0]} rows, {statistical_features_df.shape[1]} columns")

Full dataset: 1313 rows, 30 columns


## Save features

In [6]:
output_path = Path('../outputs/processed/statistical_features.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)
statistical_features_df.to_csv(output_path, index=False)

print(f"Saved to {output_path}")
print(statistical_features_df.shape)

Saved to ..\outputs\processed\statistical_features.csv
(1313, 30)


## Merge with frequency-domain features

Joins on `(subject_id, window_index)` -- `start_time` alone isn't a unique key, since it resets to 0 within every phase occurrence (e.g. 'Relax' recurs 4 times per subject); `window_index` is the 0-based position in each subject's window sequence and is unique as long as both feature sets were built with the same window/step parameters, which they are. `label`/`start_time` are dropped from the statistical side first since they'd otherwise duplicate the frequency side's identical columns.

In [7]:
frequency_features_df = pd.read_csv('../outputs/processed/frequency_features.csv')

combined_features_df = frequency_features_df.merge(
    statistical_features_df.drop(columns=['label', 'start_time']),
    on=['subject_id', 'window_index'], how='inner', validate='one_to_one',
)

print(f"frequency_features : {frequency_features_df.shape}")
print(f"statistical_features: {statistical_features_df.shape}")
print(f"combined_features   : {combined_features_df.shape}")
assert combined_features_df.shape[0] == frequency_features_df.shape[0] == statistical_features_df.shape[0], \
    "Merge dropped or duplicated rows -- window keys don't line up between the two feature sets"

combined_output_path = Path('../outputs/processed/combined_features.csv')
combined_features_df.to_csv(combined_output_path, index=False)
print(f"Saved to {combined_output_path}")

frequency_features : (1313, 29)
statistical_features: (1313, 30)
combined_features   : (1313, 55)
Saved to ..\outputs\processed\combined_features.csv
